# DestinE ClimateDT Data Example

The Climate Change Adaptation Digital Twin (Climate DT) is the first ever attempt to produce multi-decadal climate projections operationally.

The Climate DT provides multi-decadal global climate simulations with local granularity, including information specific to the sectors most affected by climate change such as renewable energy, urban planning or hydrology. The objective is to produce updated simulations every year or less, compared to the current models, ran only every several years. This enables the inclusion of the latest developments in Earth system science and digital infrastructure.

The aim of this notebook is to provide some instructions on how to request access to the ClimateDT dataset and some examples on how to retrieve data.

## Requesting access to the data

Data from ClimateDT is available through the [Destination Earth Service Platform (DESP)](https://platform.destine.eu/).

Three steps are required in order to be able to follow this notebook and download data from the DESP

### Creating a DESP accoun

Users need to register at the DESP. The following [link](https://auth.destine.eu/realms/desp/protocol/openid-connect/auth) can be used.

### Asking for upgraded access

Users from academia and research need to ask for a upgraded access in order to use the Polytope service to access the data from Destination Earth. This can be requested in the following [link](https://platform.destine.eu/access-policy-upgrade/)

This process needs manual validation and may take some time.

### Generating the Polytope token.

Once upgraded access is granted the user needs to generate the Polytpe token that is used to authnticate the polytope requests. The token is generated by executing the following script: [desp_authentication.py](https://github.com/destination-earth-digital-twins/polytope-examples/blob/main/desp-authentication.py). A copy of the script has been added to the repository for convenience.

The script will ask for your DESP username and password and will generate the token file at `${HOME}/.polytopeapirc`.


## Downloading DesinE data with Polytope

The recommended way of getting the data is through `polytope`, interfaced with `earthkit-data`. This packages must be installed since they were not included in the `uv` environment provided in the original repo.




## Downloading with earthkit

Data request to polytope are done with a MARS-like request syntax.

For more information about the **MARS keys needed** check [Data Structure and Keys](https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide&doc_page=/models/IFS-NEMO/index.html)

For more information about the **available data variables** check [Data Catalogue](https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide&doc_page=/models/IFS-NEMO/index.html)

In the following example, 1 day of an hourly field (`2t`, 2 metre temperature) is downloaded.

In [ ]:
import earthkit.data

request = {
    'class': 'd1',  # Always d1 for DestinE data
    'dataset': 'climate-dt',  # Always climate-dt for ClimateDT data
    'experiment': 'ssp3-7.0',  # Depends of the experiment. Common options: cont, hist, ssp3-7.0, tplus2.0k...
    'activity': 'projections',  # projections for dates after 20150101, baseline for dates before 20150101
    'model': 'ifs-nemo',  # Available models: ifs-nemo, ifs-fesom, icon
    'generation': '2',   # Always 2 for DestinE generation 2 data.
    'realization': '1',  # Selects between members on an ensemble
    'stream': 'clte',  # clte for high frequency data (hourly for atmos, daily for ocean). clmn for monthly means
    'resolution': 'high',  # high for highest available resolution (H1024 or H512 depending on the experiment), standard for data interpolated to H128
    'type': 'fc',  # Always fc for ClimateDT data
    'expver': '0001',  # Always 0001 for production data
    'levtype': 'sfc',  # sfc: surface, pl: pressure levels, hl: height levels, sol: soil levels, o2d: ocean 2D, o3d: ocean 3D
    'param': '2t',  # Variable by GRIB short name or GRIB paramID (both are allowed)
    'date': '20200101',  # Specify date to request in YYYYMMDD format
    'time': '0000/to/2300/by/0100'  # Specify time to request in hhmm format
}

data_1hourly = earthkit.data.from_source(
   "polytope",  # Underlying access engine
   "destination-earth",  # Select dataset
   request,
   stream=False,
   address="polytope.mn5.apps.dte.destination-earth.eu"  # Point to Marenostrum5 DataBridge
)

Data is returned in an earthkit handle, that can be then decoded to xaray

In [ ]:
data_1hourly.to_xarray()

For 3D variables (like pressure levels), an additional, `levelist` key is needed in the request

In [ ]:
import earthkit.data

request = {
    'class': 'd1',  # Always d1 for DestinE data
    'dataset': 'climate-dt',  # Always climate-dt for ClimateDT data
    'experiment': 'ssp3-7.0',  # Depends of the experiment. Common options: cont, hist, ssp3-7.0, tplus2.0k...
    'activity': 'projections',  # projections for dates after 20150101, baseline for dates before 20150101
    'model': 'ifs-nemo',  # Available models: ifs-nemo, ifs-fesom, icon
    'generation': '2',   # Always 2 for DestinE generation 2 data.
    'realization': '1',  # Selects between members on an ensemble
    'stream': 'clte',  # clte for high frequency data (hourly for atmos, daily for ocean). clmn for monthly means
    'resolution': 'high',  # high for highest available resolution (H1024 or H512 depending on the experiment), standard for data interpolated to H128
    'type': 'fc',  # Always fc for ClimateDT data
    'expver': '0001',  # Always 0001 for production data
    'levtype': 'pl',  # sfc: surface, pl: pressure levels, hl: height levels, sol: soil levels, o2d: ocean 2D, o3d: ocean 3D
    'param': 'u',  # Variable by GRIB short name or GRIB paramID (both are allowed)
    'date': '20200101',  # Specify date to request in YYYYMMDD format
    'time': '0000/to/2300/by/0600',  # Specify time to request in hhmm format
    'levelist': ['1000', '850']  # Pressure levels in hPa. Data available in plev19 scheme
}

data_pl_6hourly = earthkit.data.from_source(
   "polytope",
   "destination-earth",
   request,
   stream=False,
   address="polytope.mn5.apps.dte.destination-earth.eu")

In [ ]:
data_pl_6hourly.to_xarray()

In [ ]:
## Download examples to explore the DestinE dataset

The following examples downloads 2 days of hourly atmosferic data at highest resolution (HEALPix-1024). All sfc variables in the portfolio are included, 34 in total.

In [ ]:
from copy import deepcopy
from pathlib import Path

import earthkit.data


# Add all sfc varaibles from the ClimateDT Data Portfolio
variables = [78, 79, 134, 136, 137, 151, 165, 166, 167, 168, 207, 235, 228141, 228164, 235020, 235021, 235031, 235033, 235034, 235035, 235036, 235037, 235038, 235039, 235040, 235041, 235042, 235043, 235049, 235050, 235051, 235052, 235053, 235055]

# Set output directory
output_dir = Path("destine_data")
output_dir.mkdir(parents=True, exist_ok=True)

# Template request for IFS-NEMO SSP3-7.0 scenario
template_request = {
    'class': 'd1',  
    'dataset': 'climate-dt',  
    'experiment': 'ssp3-7.0',
    'activity': 'projections', 
    'model': 'ifs-nemo', 
    'generation': '2',
    'realization': '1',
    'stream': 'clte',
    'resolution': 'high',
    'type': 'fc',
    'expver': '0001',
    'levtype': 'sfc',
    'param': '167',
    'date': '20200101/to/20200102',
    'time': '0000/to/2300/by/0100',
}


# Iterate through variables for smaller requests
for var in variables:

    request = deepcopy(template_request)
    request["param"] = var
    
    data_sfc_var = earthkit.data.from_source(
       "polytope",
       "destination-earth",
       request,
       stream=False,
       address="polytope.mn5.apps.dte.destination-earth.eu")

    varname = data_sfc_var.to_fieldlist().get('parameter.variable')[0]

    data_sfc_var.to_target('file', output_dir / f'{varname}.grb')